# 06 - Merge Sources

## Objetivo
Integrar las fuentes reconstruidas del proyecto en un dataset único listo para análisis posterior y modelado.

## Fuentes esperadas
- `data/interim/milking.parquet`
- `data/interim/rumination.parquet`
- `data/interim/weather.parquet`
- `data/interim/pdf_events.parquet`

## Estrategia
1. Cargar cada fuente si existe.
2. Normalizar claves (`cow_id`, `date`).
3. Agregar cada fuente a nivel diario.
4. Usar `milking_daily` como base principal cuando exista.
5. Unir:
   - rumia por `cow_id + date`
   - clima por `date`
   - eventos PDF por `cow_id + date`
6. Guardar:
   - dataset completo
   - dataset filtrado al rango donde existe rumia

In [60]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Rutas del proyecto

In [61]:
CURRENT = Path.cwd().resolve()

if (CURRENT / "data" / "interim").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data" / "interim").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise FileNotFoundError("No se encontró data/interim ni en el directorio actual ni en el padre.")

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INTERIM_DIR  :", INTERIM_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT : C:\Users\PC\Documents\GitHub\ProyectoGranja
INTERIM_DIR  : C:\Users\PC\Documents\GitHub\ProyectoGranja\data\interim
PROCESSED_DIR: C:\Users\PC\Documents\GitHub\ProyectoGranja\data\processed


## 2. Archivos esperados

In [62]:
MILKING_PATH = INTERIM_DIR / "milking.parquet"
RUMINATION_PATH = INTERIM_DIR / "rumination.parquet"
WEATHER_PATH = INTERIM_DIR / "weather.parquet"
PDF_EVENTS_PATH = INTERIM_DIR / "events.parquet"

for p in [MILKING_PATH, RUMINATION_PATH, WEATHER_PATH, PDF_EVENTS_PATH]:
    print(f"{p.name:22} -> {'OK' if p.exists() else 'MISSING'}")

milking.parquet        -> OK
rumination.parquet     -> OK
weather.parquet        -> OK
events.parquet         -> OK


## 3. Funciones auxiliares

In [63]:
def LoadParquetIfExists(path: Path):
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {path.name:22} -> {df.shape}")
        return df
    print(f"Skipped {path.name:21} -> file not found")
    return None


def NormalizeCowId(df: pd.DataFrame):
    df = df.copy()
    candidate_cols = ["cow_id", "animal_id", "vid", "resolved_vid", "resolved_cow_id"]
    for col in candidate_cols:
        if col in df.columns:
            df["cow_id"] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
            return df
    return df


def EnsureDateColumn(df: pd.DataFrame, datetime_candidates, out_col="date"):
    df = df.copy()

    if out_col in df.columns:
        df[out_col] = pd.to_datetime(df[out_col], errors="coerce").dt.normalize()
        return df

    for col in datetime_candidates:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            df[out_col] = df[col].dt.normalize()
            return df

    raise KeyError(f"No se pudo crear '{out_col}'. Columnas candidatas no encontradas: {datetime_candidates}")


def ToMinutesFromHHMM(series: pd.Series):
    s = series.astype(str).str.strip()
    td = pd.to_timedelta(s, errors="coerce")
    return td.dt.total_seconds() / 60


def CoerceBooleanLikeToNumeric(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": 1, "false": 0,
        "yes": 1, "no": 0,
        "si": 1, "sí": 1,
        "x": 1, "": np.nan,
        "nan": np.nan, "none": np.nan
    }
    mapped = s.map(mapping)
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.combine_first(numeric)

## 4. Agregación diaria por fuente

In [64]:
def DailyAggMilking(df: pd.DataFrame):
    df = df.copy()
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["hora_inicio", "timestamp", "datetime", "date"])

    # Conversión de intervalos si existen
    if "duracion_mmss" in df.columns and "duracion_min" not in df.columns:
        df["duracion_min"] = ToMinutesFromHHMM(df["duracion_mmss"])

    if "intervalo_ordeno_hhmm" in df.columns and "intervalo_ordeno_min" not in df.columns:
        df["intervalo_ordeno_min"] = ToMinutesFromHHMM(df["intervalo_ordeno_hhmm"])

    numeric_candidates = [
        "produccion_kg", "di", "dd", "ti", "td",
        "duracion_min", "intervalo_ordeno_min",
        "numero_ordeno"
    ]

    for c in numeric_candidates:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    event_like_cols = ["patada", "incompleto", "pezones_no_encontrados"]
    for c in event_like_cols:
        if c in df.columns:
            df[c] = CoerceBooleanLikeToNumeric(df[c])

    agg = {}
    if "produccion_kg" in df.columns: agg["produccion_kg"] = "sum"
    if "di" in df.columns: agg["di"] = "sum"
    if "dd" in df.columns: agg["dd"] = "sum"
    if "ti" in df.columns: agg["ti"] = "sum"
    if "td" in df.columns: agg["td"] = "sum"
    if "numero_ordeno" in df.columns: agg["numero_ordeno"] = "count"
    if "duracion_min" in df.columns: agg["duracion_min"] = "sum"
    if "intervalo_ordeno_min" in df.columns: agg["intervalo_ordeno_min"] = "mean"
    if "patada" in df.columns: agg["patada"] = "sum"
    if "incompleto" in df.columns: agg["incompleto"] = "sum"
    if "pezones_no_encontrados" in df.columns: agg["pezones_no_encontrados"] = "sum"

    for c in ["ubre", "destino_leche", "ms", "source_file", "source_sheet"]:
        if c in df.columns:
            agg[c] = "first"

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg(agg)
          .reset_index()
    )

    daily = daily.rename(columns={
        "numero_ordeno": "ordenos_dia",
        "duracion_min": "duracion_total_min",
        "intervalo_ordeno_min": "intervalo_ordeno_prom_min",
        "patada": "patadas_dia",
        "incompleto": "incompletos_dia",
        "pezones_no_encontrados": "pezones_no_encontrados_dia"
    })

    return daily


def DailyAggRumination(df: pd.DataFrame):
    df = df.copy()
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["date", "timestamp", "datetime", "fecha"])

    # Unificar rumia
    if "ruminating_minutes" in df.columns and "rumia_min" not in df.columns:
        df = df.rename(columns={"ruminating_minutes": "rumia_min"})
    elif "ruminating" in df.columns and "rumia_min" not in df.columns:
        df = df.rename(columns={"ruminating": "rumia_min"})

    # Unificar group_id sin duplicarlo
    if "resolved_group" in df.columns and "group_id" in df.columns:
        df["group_id"] = df["group_id"].combine_first(df["resolved_group"])
        df = df.drop(columns=["resolved_group"])
    elif "resolved_group" in df.columns:
        df = df.rename(columns={"resolved_group": "group_id"})

    # Asegurar tipos numéricos
    for c in ["rumia_min", "days_in_milk", "lactation_age", "weekday", "month"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    agg = {}
    for c in ["rumia_min", "days_in_milk", "lactation_age", "weekday", "month"]:
        if c in df.columns:
            agg[c] = "mean"
    for c in ["group_id", "source_file"]:
        if c in df.columns:
            agg[c] = "first"

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg(agg)
          .reset_index()
    )

    return daily


def DailyAggWeather(df: pd.DataFrame):
    df = df.copy()
    df = EnsureDateColumn(df, ["time", "date", "datetime", "timestamp", "fecha"])

    rename_map = {
        "temperatura": "temperature_c",
        "temperature": "temperature_c",
        "temperature_2m": "temperature_c",
        "humedad": "humidity_pct",
        "humidity": "humidity_pct",
        "relative_humidity_2m": "humidity_pct",
        "lluvia": "rain_mm",
        "rain": "rain_mm",
        "precipitation": "rain_mm",
        "viento": "wind_speed",
        "wind": "wind_speed",
        "wind_speed_10m": "wind_speed",
        "pressure_msl": "pressure_msl"
    }

    for old, new in rename_map.items():
        if old in df.columns and old != new:
            df = df.rename(columns={old: new})

    numeric_cols = [
        c for c in [
            "temperature_c",
            "humidity_pct",
            "pressure_msl",
            "rain_mm",
            "wind_speed",
            "heat_index"
        ] if c in df.columns
    ]

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if numeric_cols:
        daily = df.groupby("date", dropna=False)[numeric_cols].mean().reset_index()
    else:
        daily = df[["date"]].drop_duplicates().copy()

    return daily


def DailyAggPdfEvents(df: pd.DataFrame):

    df = df.copy()

    # Normalizar cow_id
    if "cow_id" not in df.columns:
        if "animal_id" in df.columns:
            df["cow_id"] = pd.to_numeric(df["animal_id"], errors="coerce")

    df = EnsureDateColumn(df, ["event_date","date","fecha_evento","timestamp","datetime"])

    if "raw_event_text" not in df.columns:
        text_col = None
        for c in ["evento","descripcion","event_text","raw_text"]:
            if c in df.columns:
                text_col = c
                break

        if text_col:
            df = df.rename(columns={text_col:"raw_event_text"})
        else:
            df["raw_event_text"] = ""

    df["raw_event_text"] = df["raw_event_text"].astype(str)

    daily = (
        df.groupby(["cow_id","date"], dropna=False)
        .agg(
            eventos_pdf_count=("raw_event_text","count"),
            eventos_pdf_text=("raw_event_text", lambda s: " | ".join(s.head(10)))
        )
        .reset_index()
    )

    return daily

## 5. Cargar fuentes

In [65]:
milking_raw = LoadParquetIfExists(MILKING_PATH)
rumination_raw = LoadParquetIfExists(RUMINATION_PATH)
weather_raw = LoadParquetIfExists(WEATHER_PATH)
pdf_events_raw = LoadParquetIfExists(PDF_EVENTS_PATH)

Loaded milking.parquet        -> (23763, 17)
Loaded rumination.parquet     -> (19110, 15)
Loaded weather.parquet        -> (29376, 6)
Loaded events.parquet         -> (6890, 6)


## 6. Agregar a nivel diario

In [66]:
milking_daily = DailyAggMilking(milking_raw) if milking_raw is not None else None
rumination_daily = DailyAggRumination(rumination_raw) if rumination_raw is not None else None
weather_daily = DailyAggWeather(weather_raw) if weather_raw is not None else None
pdf_events_daily = DailyAggPdfEvents(pdf_events_raw) if pdf_events_raw is not None else None

for name, df_src in [
    ("milking_daily", milking_daily),
    ("rumination_daily", rumination_daily),
    ("weather_daily", weather_daily),
    ("pdf_events_daily", pdf_events_daily),
]:
    if df_src is None:
        print(f"{name:18} -> None")
    else:
        print(f"{name:18} -> {df_src.shape}")

milking_daily      -> (9814, 14)
rumination_daily   -> (7098, 9)
weather_daily      -> (1224, 6)
pdf_events_daily   -> (561, 4)


## 7. Vista rápida

In [67]:
if milking_daily is not None:
    display(milking_daily.head())

if rumination_daily is not None:
    display(rumination_daily.head())

if weather_daily is not None:
    display(weather_daily.head())

if pdf_events_daily is not None:
    display(pdf_events_daily.head())

,cow_id,date,produccion_kg,di,dd,ti,td,ordenos_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,source_file
0,1204,2025-01-01,16.6800,5.1600,4.6400,0.0000,6.8800,1,0.0000,NaN,0,Tanque,VMS 1,Producciones de leche1204.xls
1,1204,2025-01-02,17.9100,2.6500,4.4400,5.7500,5.0700,2,0.0000,NaN,1,Divert 3,VMS 1,Producciones de leche1204.xls
2,1204,2025-01-03,25.3600,6.2600,6.5200,2.7700,9.8100,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls
3,1204,2025-01-04,16.7100,4.1300,3.8900,2.9600,5.7300,1,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls
4,1204,2025-01-05,25.0400,5.8500,5.9400,4.8500,8.4000,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls


,cow_id,date,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,source_file
0,1204,2025-06-27,NaN,417.0000,417.0000,4.0000,6.0000,100.0000,group_100_ruminating_rumia.csv
1,1204,2025-06-28,657.0000,418.0000,418.0000,5.0000,6.0000,100.0000,group_100_ruminating_rumia.csv
2,1204,2025-06-29,495.0000,419.0000,419.0000,6.0000,6.0000,100.0000,group_100_ruminating_rumia.csv
3,1204,2025-06-30,549.0000,420.0000,420.0000,0.0000,6.0000,100.0000,group_100_ruminating_rumia.csv
4,1204,2025-07-01,566.0000,421.0000,421.0000,1.0000,7.0000,100.0000,group_100_ruminating_rumia.csv


,date,temperature_c,humidity_pct,pressure_msl,rain_mm,wind_speed
0,2022-05-26,20.4333,63.4583,"1,016.0125",0.0000,14.5917
1,2022-05-27,19.9792,61.0000,"1,016.9792",0.0125,15.1958
2,2022-05-28,19.3375,56.8750,"1,014.6958",0.0000,13.5667
3,2022-05-29,20.7167,48.7083,"1,010.5667",0.0000,8.1792
4,2022-05-30,21.5208,49.1667,"1,008.6958",0.0000,9.3875


,cow_id,date,eventos_pdf_count,eventos_pdf_text
0,1204,2023-07-27,1,"Invitación Visita 13/07/2 407), Available for ..."
1,1204,2025-05-29,1,Invitación Visita 05/06/2 Gestacion (>40 dias ...
2,1204,2025-06-05,1,Invitación Visita 19/06/2 Gestacion (>40 dias ...
3,1204,2025-06-19,1,"Control de Gest 19/06/2 User1 19/06/2025, Diag..."
4,1204,2025-07-24,1,"Control de Gest 24/07/2 User1 24/07/2025, Reco..."


## 8. Diagnóstico de traslape de claves

In [68]:
print("=== KEY OVERLAP DIAGNOSTIC ===")

if milking_daily is not None and rumination_daily is not None:
    milking_ids = set(milking_daily["cow_id"].dropna().unique())
    rumination_ids = set(rumination_daily["cow_id"].dropna().unique())

    print("Milking cows:", len(milking_ids))
    print("Rumination cows:", len(rumination_ids))
    print("Shared cows:", len(milking_ids & rumination_ids))

    milking_keys = set(zip(milking_daily["cow_id"], milking_daily["date"]))
    rumination_keys = set(zip(rumination_daily["cow_id"], rumination_daily["date"]))

    print("Shared cow-date keys:", len(milking_keys & rumination_keys))

if milking_daily is not None and pdf_events_daily is not None:
    milking_ids = set(milking_daily["cow_id"].dropna().unique())
    pdf_ids = set(pdf_events_daily["cow_id"].dropna().unique())

    print("\nMilking cows:", len(milking_ids))
    print("PDF cows:", len(pdf_ids))
    print("Shared cows:", len(milking_ids & pdf_ids))

    milking_keys = set(zip(milking_daily["cow_id"], milking_daily["date"]))
    pdf_keys = set(zip(pdf_events_daily["cow_id"], pdf_events_daily["date"]))

    print("Shared cow-date keys:", len(milking_keys & pdf_keys))

=== KEY OVERLAP DIAGNOSTIC ===
Milking cows: 65
Rumination cows: 78
Shared cows: 64
Shared cow-date keys: 28

Milking cows: 65
PDF cows: 67
Shared cows: 65
Shared cow-date keys: 148


## 9. Rangos temporales

In [69]:
if milking_daily is not None:
    print("Milking range   :", milking_daily["date"].min(), "->", milking_daily["date"].max())
if rumination_daily is not None:
    print("Rumination range:", rumination_daily["date"].min(), "->", rumination_daily["date"].max())
if pdf_events_daily is not None:
    print("PDF range       :", pdf_events_daily["date"].min(), "->", pdf_events_daily["date"].max())
if weather_daily is not None:
    print("Weather range   :", weather_daily["date"].min(), "->", weather_daily["date"].max())

Milking range   : 2025-01-01 00:00:00 -> 2025-09-30 00:00:00
Rumination range: 2025-06-27 00:00:00 -> 2025-09-25 00:00:00
PDF range       : 2022-05-26 00:00:00 -> 2025-09-25 00:00:00
Weather range   : 2022-05-26 00:00:00 -> 2025-09-30 00:00:00


## 10. Elegir base del merge

Se usa `milking_daily` como base principal cuando exista.

In [70]:
if milking_daily is not None:
    merged = milking_daily.copy()
    print("Base del merge: milking_daily")
elif rumination_daily is not None:
    merged = rumination_daily.copy()
    print("Base del merge: rumination_daily")
else:
    raise ValueError("Se requiere al menos milking.parquet o rumination.parquet para construir la base del merge.")

Base del merge: milking_daily


## 11. Merge con rumia

In [71]:
if rumination_daily is not None and merged is not rumination_daily:
    merged = merged.merge(
        rumination_daily,
        on=["cow_id", "date"],
        how="left",
        suffixes=("", "_rum")
    )

print("Shape after rumination merge:", merged.shape)
display(merged.head())

Shape after rumination merge: (9814, 21)


,cow_id,date,produccion_kg,di,dd,ti,td,ordenos_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,source_file,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,source_file_rum
0,1204,2025-01-01,16.6800,5.1600,4.6400,0.0000,6.8800,1,0.0000,NaN,0,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1204,2025-01-02,17.9100,2.6500,4.4400,5.7500,5.0700,2,0.0000,NaN,1,Divert 3,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1204,2025-01-03,25.3600,6.2600,6.5200,2.7700,9.8100,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1204,2025-01-04,16.7100,4.1300,3.8900,2.9600,5.7300,1,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1204,2025-01-05,25.0400,5.8500,5.9400,4.8500,8.4000,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 12. Merge con clima

In [72]:
if weather_daily is not None:
    merged = merged.merge(
        weather_daily,
        on="date",
        how="left",
        suffixes=("", "_weather")
    )

print("Shape after weather merge:", merged.shape)
display(merged.head())

Shape after weather merge: (9814, 26)


,cow_id,date,produccion_kg,di,dd,ti,td,ordenos_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,source_file,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,source_file_rum,temperature_c,humidity_pct,pressure_msl,rain_mm,wind_speed
0,1204,2025-01-01,16.6800,5.1600,4.6400,0.0000,6.8800,1,0.0000,NaN,0,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.2667,43.2917,"1,019.2833",0.0000,11.1500
1,1204,2025-01-02,17.9100,2.6500,4.4400,5.7500,5.0700,2,0.0000,NaN,1,Divert 3,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.4042,64.9167,"1,022.5042",0.0000,13.1875
2,1204,2025-01-03,25.3600,6.2600,6.5200,2.7700,9.8100,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.7917,73.0000,"1,023.9875",0.0000,16.0458
3,1204,2025-01-04,16.7100,4.1300,3.8900,2.9600,5.7300,1,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.3333,70.7500,"1,020.8167",0.0292,7.8000
4,1204,2025-01-05,25.0400,5.8500,5.9400,4.8500,8.4000,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.3333,58.4583,"1,018.5042",0.0000,8.4458


## 13. Merge con eventos PDF

In [73]:
if pdf_events_daily is not None:
    merged = merged.merge(
        pdf_events_daily,
        on=["cow_id", "date"],
        how="left",
        suffixes=("", "_pdf")
    )

print("Shape after pdf merge:", merged.shape)
display(merged.head())

Shape after pdf merge: (9814, 28)


,cow_id,date,produccion_kg,di,dd,ti,td,ordenos_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,source_file,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,source_file_rum,temperature_c,humidity_pct,pressure_msl,rain_mm,wind_speed,eventos_pdf_count,eventos_pdf_text
0,1204,2025-01-01,16.6800,5.1600,4.6400,0.0000,6.8800,1,0.0000,NaN,0,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.2667,43.2917,"1,019.2833",0.0000,11.1500,NaN,NaN
1,1204,2025-01-02,17.9100,2.6500,4.4400,5.7500,5.0700,2,0.0000,NaN,1,Divert 3,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.4042,64.9167,"1,022.5042",0.0000,13.1875,NaN,NaN
2,1204,2025-01-03,25.3600,6.2600,6.5200,2.7700,9.8100,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.7917,73.0000,"1,023.9875",0.0000,16.0458,NaN,NaN
3,1204,2025-01-04,16.7100,4.1300,3.8900,2.9600,5.7300,1,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.3333,70.7500,"1,020.8167",0.0292,7.8000,NaN,NaN
4,1204,2025-01-05,25.0400,5.8500,5.9400,4.8500,8.4000,2,0.0000,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.3333,58.4583,"1,018.5042",0.0000,8.4458,NaN,NaN


## 14. Variables temporales mínimas

In [74]:
merged["date"] = pd.to_datetime(merged["date"], errors="coerce")
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day
merged["weekday_date"] = merged["date"].dt.weekday

## 15. Reporte de nulos del dataset completo

In [75]:
sort_cols = [c for c in ["cow_id", "date"] if c in merged.columns]
if sort_cols:
    merged = merged.sort_values(sort_cols).reset_index(drop=True)

nan_report = pd.DataFrame({
    "column": merged.columns,
    "nan_count": merged.isna().sum().values,
    "nan_pct": (merged.isna().mean() * 100).values
}).sort_values("nan_pct", ascending=False)

print("Shape merged completo:", merged.shape)
display(nan_report.head(30))

Shape merged completo: (9814, 31)


,column,nan_count,nan_pct
9,intervalo_ordeno_prom_min,9814,100.0000
14,rumia_min,9787,99.7249
20,source_file_rum,9786,99.7147
19,group_id,9786,99.7147
17,weekday,9786,99.7147
16,lactation_age,9786,99.7147
15,days_in_milk,9786,99.7147
26,eventos_pdf_count,9666,98.4920
27,eventos_pdf_text,9666,98.4920
6,td,0,0.0000


## 16. Crear dataset filtrado al rango de rumia

Como la rumia tiene cobertura temporal más corta, esta versión sirve mejor para modelado cuando se quiera usar esa fuente.

In [76]:
if rumination_daily is not None:
    rum_min = rumination_daily["date"].min()
    rum_max = rumination_daily["date"].max()

    merged_model = merged[
        (merged["date"] >= rum_min) &
        (merged["date"] <= rum_max)
    ].copy()

    print("Rango rumia:", rum_min, "->", rum_max)
else:
    merged_model = merged.copy()

print("Shape merged completo:", merged.shape)
print("Shape merged_model   :", merged_model.shape)

if "rumia_min" in merged_model.columns:
    rumia_cov = merged_model["rumia_min"].notna().mean() * 100
    print(f"Cobertura de rumia en merged_model: {rumia_cov:.2f}%")

Rango rumia: 2025-06-27 00:00:00 -> 2025-09-25 00:00:00
Shape merged completo: (9814, 31)
Shape merged_model   : (28, 31)
Cobertura de rumia en merged_model: 96.43%


## 17. Reporte de nulos del dataset para modelado

In [77]:
nan_report_model = pd.DataFrame({
    "column": merged_model.columns,
    "nan_count": merged_model.isna().sum().values,
    "nan_pct": (merged_model.isna().mean() * 100).values
}).sort_values("nan_pct", ascending=False)

display(nan_report_model.head(30))

,column,nan_count,nan_pct
9,intervalo_ordeno_prom_min,28,100.0000
26,eventos_pdf_count,26,92.8571
27,eventos_pdf_text,26,92.8571
14,rumia_min,1,3.5714
4,dd,0,0.0000
3,di,0,0.0000
0,cow_id,0,0.0000
6,td,0,0.0000
5,ti,0,0.0000
8,duracion_total_min,0,0.0000


## 18. Validaciones rápidas

In [78]:
print("=== VALIDACIÓN FINAL ===")
print("Merged completo:")
print("  filas:", merged.shape[0])
print("  columnas:", merged.shape[1])

if "cow_id" in merged.columns:
    print("  vacas:", merged["cow_id"].nunique(dropna=True))
print("  rango fechas:", merged["date"].min(), "->", merged["date"].max())

if "produccion_kg" in merged.columns:
    print("  producción total:", merged["produccion_kg"].sum())

if "rumia_min" in merged_model.columns:
    print("\nMerged model:")
    print("  filas:", merged_model.shape[0])
    print("  vacas:", merged_model["cow_id"].nunique(dropna=True))
    print("  rumia total:", merged_model["rumia_min"].sum(skipna=True))

=== VALIDACIÓN FINAL ===
Merged completo:
  filas: 9814
  columnas: 31
  vacas: 65
  rango fechas: 2025-01-01 00:00:00 -> 2025-09-30 00:00:00
  producción total: 368634.65

Merged model:
  filas: 28
  vacas: 2
  rumia total: 7036.0


## 19. Guardar resultados

In [79]:
OUTPUT_FULL = PROCESSED_DIR / "training_dataset.parquet"
OUTPUT_FULL_CSV = PROCESSED_DIR / "training_dataset.csv"

OUTPUT_MODEL = PROCESSED_DIR / "training_dataset_with_rumination.parquet"
OUTPUT_MODEL_CSV = PROCESSED_DIR / "training_dataset_with_rumination.csv"

merged.to_parquet(OUTPUT_FULL, index=False)
merged.to_csv(OUTPUT_FULL_CSV, index=False)

merged_model.to_parquet(OUTPUT_MODEL, index=False)
merged_model.to_csv(OUTPUT_MODEL_CSV, index=False)

print("Saved:")
print(" -", OUTPUT_FULL)
print(" -", OUTPUT_FULL_CSV)
print(" -", OUTPUT_MODEL)
print(" -", OUTPUT_MODEL_CSV)

Saved:
 - C:\Users\PC\Documents\GitHub\ProyectoGranja\data\processed\training_dataset.parquet
 - C:\Users\PC\Documents\GitHub\ProyectoGranja\data\processed\training_dataset.csv
 - C:\Users\PC\Documents\GitHub\ProyectoGranja\data\processed\training_dataset_with_rumination.parquet
 - C:\Users\PC\Documents\GitHub\ProyectoGranja\data\processed\training_dataset_with_rumination.csv


## 20. Próximo paso
Con este notebook ya puedes pasar a:
- `07_feature_engineering.ipynb`
- `08_model_training.ipynb`

La versión `training_dataset_with_rumination.parquet` es la más apropiada si el modelo necesita usar rumia.


En este momento el conteido de la información de rumia es demasiado limitado, por lo que training_dataset_with_rumiation es muy pequeño, se recomienda agregar mas datos de rumia para poderlo utilizar